In [28]:
with open('input.txt','r', encoding='utf-8') as f:
    text = f.read()

In [29]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [30]:
# Here we find all of the unique characters that occur in our input data
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [31]:
# Tokenizer (gives you the mapping of string to integer), this is a very simple one. 
# There are other tokenizers like SentencePiece which tokenazes not single words or characters but sub words
# ChatGPT uses TikToken which has 50257 tokens

# Now we create a mapping from characters to integers
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]  # Encoder here takes a string and outputs a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # Decoder here takes a list of integers and outputs a string

print(encode("Nikola Vukas"))
print(decode(encode("Nikola Vukas")))

[26, 47, 49, 53, 50, 39, 1, 34, 59, 49, 39, 57]
Nikola Vukas


In [32]:
# Encoding the entire input.txt

import torch 
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # Identical translation which represents the text as a sequence of integers

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [33]:
# Train validation split

n = int(0.9*len(data)) # 90% will be test data and 10% is validation to avoid overfitting
train_data = data[:n]
val_data = data[n:]

In [34]:
# When training we train on chunks of data, not on the entire data set, this is called block_size, can be other

block_size = 8
# We print 9 because in this context we can make 8 predictions
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [35]:
# Here we show the way predictions are made for each context
# This is also used so the model knows how to predict with smaller contexts and not only full block size, not just for efficiency
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When input is {context} the target is: {target}")


When input is tensor([18]) the target is: 47
When input is tensor([18, 47]) the target is: 56
When input is tensor([18, 47, 56]) the target is: 57
When input is tensor([18, 47, 56, 57]) the target is: 58
When input is tensor([18, 47, 56, 57, 58]) the target is: 1
When input is tensor([18, 47, 56, 57, 58,  1]) the target is: 15
When input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is: 47
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is: 58


In [36]:
torch.manual_seed(1337)

block_size = 8 # This tells you what the maximum context length is for predictions
batch_size = 4 # This tells you how many independent context's we want to run in parallel

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size, (batch_size,)) # (batch_size,) generates 4 numbers since the batch size is 4!
    x = torch.stack([data[i:i+block_size] for i in ix]) # Here we stack the rows 
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

xb,yb = get_batch('train')
print('Inputs: ')
print(xb.shape)
print(xb) # Here we see that a row represents a context and there are 4 since that is what we defined as the batch size
print('Targets: ')
print(yb.shape)
print(yb)

print('====')

for b in range(batch_size): # Batch dimension
    for t in range(block_size): # Time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f'When input is: {context.tolist()} the target: {target}')

Inputs: 
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
Targets: 
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
====
When input is: [24] the target: 43
When input is: [24, 43] the target: 58
When input is: [24, 43, 58] the target: 5
When input is: [24, 43, 58, 5] the target: 57
When input is: [24, 43, 58, 5, 57] the target: 1
When input is: [24, 43, 58, 5, 57, 1] the target: 46
When input is: [24, 43, 58, 5, 57, 1, 46] the target: 43
When input is: [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
When input is: [44] the target: 53
When input is: [44, 53] the target: 56
When input is: [44, 53, 56] the target: 1
When input is: [44, 53, 56, 1] the target: 58
When input is: [44, 53, 56, 1, 58] the target: 46
When i

## 🔑 Embeddings vs Logits (Bigram Toy vs Real LLM)

In a normal LLM, the embedding matrix and the output (logit) matrix are two separate things: the embedding matrix of shape `(vocab_size, d_model)` represents each token as a vector in some abstract space, while the output matrix of shape `(d_model, vocab_size)` maps hidden states to scores for each token in the vocabulary, meaning embeddings are not probabilities. In the toy Bigram model, however, this distinction collapses because the embedding table is defined as `nn.Embedding(vocab_size, vocab_size)`, making the embedding dimension equal to the vocabulary size. As a result, each row of the embedding table is already aligned with the output logits, so the “embedding” of a token is effectively a logit vector, which only becomes probabilities after applying softmax. During text generation, the model takes the last token in the sequence, retrieves its logit vector, applies softmax to get probabilities, and then samples the next token. The key takeaway is that in a real LLM, embeddings and logits are different, but in this simplified Bigram model, they are the same, which is why it feels like embeddings are directly giving probabilities.



In [ ]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)
        
    def forward(self,idx,targets = None):
        
        #idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C(channels or vocab size))
        
        # Embeddings = how tokens are represented going into the network.

        #Logits = the network’s raw scores for predicting the next token, before softmax. This is what comes as an output of the LLM
        if targets is None: 
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T,C) # Here we basically stretch out the array so it's 2d and C is second dimension so that cross_entropy works
            targets = targets.view(B*T)
            
            loss = F.cross_entropy(logits,targets) # Negative log loss. IT WANTS B,C,T! So we have to change the shape of the logit
        return logits,loss
    
    def generate(self,idx,max_new_tokens):
        for _ in range(max_new_tokens): 
            # Get the prediction
            logits,loss = self(idx)
            # Focus only on the last step - instead of the (B,T,C)
            logits = logits[:,-1,:] # We pluck out only (B,C)
            # Apply softmax to get probabilities
            probs = F.softmax(logits,dim=1)
            # Sample from the distribution
            idx_next = torch.multinomial(probs,num_samples=1) # (B,1)
            # Append the sampled intex to the sequence
            idx = torch.cat((idx,idx_next), dim=1) # (B,T+1)
        return idx
    
m = BigramLanguageModel(vocab_size)
logits, loss= m(xb,yb)
print(logits.shape)  # We have 4 sequences (batch), each with 8 tokens, and every token is replaced by the same vector of size vocab_size which was defined in the Embeedding(it's not always the same as the number of tokens).

print(loss) # Look at log loss again - Cross Entropy looks at the difference in the prob. distributions 
# Why [0].tolist()
print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long),max_new_tokens = 100)[0].tolist())) # Feed in a 1,1 tensor of 0 which is the new line token
# We get garbage because it is not trained

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

Sr?qP-QWktXoL&jLDJgOLVz'RIoDqHdhsV&vLLxatjscMpwLERSPyao.qfzs$Ys$zF-w,;eEkzxjgCKFChs!iWW.ObzDnxA Ms$3


During text generation, the model works in a loop where at each step it predicts only the next token. First, the forward pass produces logits of shape `(B, T, C)`, where `B` is the batch size, `T` is the current sequence length, and `C` is the vocabulary size. Since we only need the prediction for the most recent token, we select the last time step to get logits of shape `(B, C)`. These logits are passed through a softmax to obtain probabilities over the vocabulary, and then a token is sampled from this distribution using `torch.multinomial`, giving the next token `(B, 1)`. This sampled token is concatenated to the existing sequence, extending it to `(B, T+1)`. The process repeats until the desired number of tokens (`max_new_tokens`) is generated, gradually building the output sequence one token at a time.


In [51]:
optimizer = torch.optim.AdamW(m.parameters(),lr=1e-3)

In [ ]:
batch_size = 32
for steps in range(10000):
    
    # Sample a batch of data
    xb,yb = get_batch('train')
    
    # Evaluate the loss
    logits,loss = m(xb,yb)
    optimizer.zero_grad(set_to_none=True) # Remove gradients before the next step
    loss.backward()
    optimizer.step()
    
print(loss.item())    


2.4033024311065674


In [60]:
print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long),max_new_tokens = 300)[0].tolist()))




I wit an, s m d tot fou s; orotras s my omaillon he ll as grut wer bun.
TRDid ERUThele goot, d lpou bezed are'
GBe titheeno torde m:

Fe ois; isofu, amake, thr bimyoomo, h
My m, tak is nd p w we ino t way uke, e ng s?
INuristancat msaksu my;
The y hanormy.
I d de m ingacowolakngent.
Pospundourmast
